# DeepAssimilate: 3-Step Quickstart

This notebook demonstrates the complete DeepAssimilate workflow on synthetic data:

1. **Step 1 - Architecture Search (NAS)**: Find the best diffusion model architecture
2. **Step 2 - Train Unconditional Diffusion Model**: Train the winner
3. **Step 3 - Score-Based Data Assimilation**: Assimilate sparse observations

Uses synthetic Gaussian blob data so it runs in ~2 minutes on any machine.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import deepassimilate as da

print(f"deepassimilate v{da.__version__}")
device = da.get_device()
print(f"Device: {device}")

## Create Synthetic Data

We create smooth Gaussian blob fields as a stand-in for weather data. Each sample is a 32x32 grid with 1-3 blobs at random positions.

In [ ]:
np.random.seed(42)
N, H, W = 200, 32, 32
data = np.zeros((N, H, W), dtype=np.float32)

x_grid, y_grid = np.meshgrid(np.linspace(-3, 3, W), np.linspace(-3, 3, H))

for i in range(N):
    n_blobs = np.random.randint(1, 4)
    for _ in range(n_blobs):
        cx, cy = np.random.uniform(-2, 2, 2)
        amp = np.random.uniform(0.5, 1.5)
        width = np.random.uniform(0.5, 1.5)
        data[i] += amp * np.exp(-((x_grid - cx)**2 + (y_grid - cy)**2) / width**2)
    data[i] += 0.05 * np.random.randn(H, W)

train_ds = da.WeatherDataset(data[:160])
val_ds = da.WeatherDataset(data[160:])

# Plot some samples
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, ax in enumerate(axes):
    ax.imshow(data[i], cmap='viridis')
    ax.set_title(f'Sample {i}')
    ax.axis('off')
plt.suptitle('Synthetic training data (Gaussian blobs)', fontsize=14)
plt.tight_layout()
plt.show()

## Step 1: Architecture Search (NAS)

Autoresearch-style search: train each candidate for a fixed time budget, keep the best. We use a small search space and short budget for this demo.

In [ ]:
nas_cfg = da.NASConfig(
    time_budget_seconds=15,       # 15s per candidate (use 300 for real runs)
    max_experiments=5,
    img_size=H,
    in_channels=1,
    batch_size=16,
    scheduler='ddpm',
    results_dir='nas_results',
    search_space=[
        {'name': 'tiny_2block', 'block_out_channels': (32, 64),
         'down_block_types': ('DownBlock2D',)*2, 'up_block_types': ('UpBlock2D',)*2,
         'layers_per_block': 1},
        {'name': 'small_3block', 'block_out_channels': (32, 64, 128),
         'down_block_types': ('DownBlock2D',)*3, 'up_block_types': ('UpBlock2D',)*3,
         'layers_per_block': 1},
        {'name': 'medium_3block', 'block_out_channels': (64, 128, 256),
         'down_block_types': ('DownBlock2D',)*3, 'up_block_types': ('UpBlock2D',)*3,
         'layers_per_block': 2},
        {'name': 'attn_3block', 'block_out_channels': (64, 128, 256),
         'down_block_types': ('DownBlock2D', 'AttnDownBlock2D', 'DownBlock2D'),
         'up_block_types': ('UpBlock2D', 'AttnUpBlock2D', 'UpBlock2D'),
         'layers_per_block': 2},
        {'name': 'wide_2block', 'block_out_channels': (128, 256),
         'down_block_types': ('DownBlock2D',)*2, 'up_block_types': ('UpBlock2D',)*2,
         'layers_per_block': 2},
    ],
)

nas_result = da.search_architecture(train_ds, val_ds, cfg=nas_cfg)
print(f"\nWinner: {nas_result.best_config['name']}")
print(f"Val loss: {nas_result.best_val_loss:.6f}")

In [ ]:
# Plot NAS results
import pandas as pd

results_df = pd.read_csv(nas_result.results_tsv, sep='\t')
colors = {'keep': 'green', 'discard': 'red', 'crash': 'gray'}
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Val loss by architecture
bars = axes[0].bar(results_df['name'], results_df['val_loss'],
                   color=[colors[s] for s in results_df['status']])
axes[0].set_ylabel('Validation MSE Loss')
axes[0].set_title('Step 1: NAS Results')
axes[0].tick_params(axis='x', rotation=30)
axes[0].axhline(nas_result.best_val_loss, color='green', ls='--', alpha=0.5, label='Best')
axes[0].legend()

# Params vs loss
axes[1].scatter(results_df['num_params'], results_df['val_loss'],
                c=[colors[s] for s in results_df['status']], s=100, zorder=5)
for _, row in results_df.iterrows():
    axes[1].annotate(row['name'], (row['num_params'], row['val_loss']),
                     fontsize=8, ha='center', va='bottom')
axes[1].set_xlabel('Number of Parameters')
axes[1].set_ylabel('Validation MSE Loss')
axes[1].set_title('Params vs Performance')

plt.tight_layout()
plt.show()

## Step 2: Train the Best Architecture

Now we train the winning architecture for longer. We use the config from NAS and build the model directly.

In [ ]:
# Build model from NAS winner
from diffusers import UNet2DModel

best_cfg = {k: v for k, v in nas_result.best_config.items() if k != 'name'}
model = UNet2DModel(
    sample_size=H, in_channels=1, out_channels=1, **best_cfg
).to(device)
print(f"Model: {nas_result.best_config['name']}, params: {da.count_parameters(model):,}")

# Build scheduler
scheduler = da.build_scheduler('ddpm', num_train_timesteps=1000)

# Training loop
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
loader = torch.utils.data.DataLoader(train_ds, batch_size=16, shuffle=True)

num_epochs = 30  # Use more epochs for real data (100+)
train_losses = []

model.train()
for epoch in range(num_epochs):
    epoch_losses = []
    for batch in loader:
        x = batch.to(device)
        t = torch.randint(0, 1000, (x.shape[0],), device=device).long()
        noise = torch.randn_like(x)
        noisy = scheduler.add_noise(x, noise, t)
        pred = model(noisy, t).sample
        loss = torch.nn.functional.mse_loss(pred, noise)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        epoch_losses.append(loss.item())

    avg = np.mean(epoch_losses)
    train_losses.append(avg)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}: loss={avg:.4f}")

# Plot training curve
plt.figure(figsize=(8, 4))
plt.plot(train_losses)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Step 2: Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Unconditional Sampling

Generate samples from the trained model (no observations) to verify it learned the distribution.

In [ ]:
# Unconditional sampling
model.eval()
scheduler.set_timesteps(50, device=device)

samples = torch.randn(4, 1, H, W, device=device)
with torch.no_grad():
    for t in scheduler.timesteps:
        pred = model(samples, t.expand(4)).sample
        samples = scheduler.step(pred, t, samples).prev_sample

# Denormalize
samples_np = samples.cpu().numpy()
samples_dn = samples_np * (train_ds.max - train_ds.min) + train_ds.min

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    axes[0, i].imshow(data[i], cmap='viridis')
    axes[0, i].set_title(f'Real {i}')
    axes[0, i].axis('off')
    axes[1, i].imshow(samples_dn[i, 0], cmap='viridis')
    axes[1, i].set_title(f'Generated {i}')
    axes[1, i].axis('off')
axes[0, 0].set_ylabel('Real', fontsize=14)
axes[1, 0].set_ylabel('Generated', fontsize=14)
plt.suptitle('Unconditional Samples vs Real Data', fontsize=14)
plt.tight_layout()
plt.show()

## Step 3: Score-Based Data Assimilation

Now the key step: combine the trained diffusion prior with sparse observations. We observe only 10% of grid points and reconstruct the full field.

In [ ]:
# Take a test sample as ground truth
truth = val_ds[0].unsqueeze(0).to(device)  # [1, 1, H, W]

# Create sparse observations (10% observed)
obs_mask = da.make_random_mask((1, 1, H, W), obs_fraction=0.10, seed=123)
observations = truth.clone()
observations[~obs_mask.to(device)] = float('nan')

print(f"Observed fraction: {obs_mask.float().mean():.2%}")
print(f"Observed points: {obs_mask.sum().item()} / {H*W}")

# Run DA - this is the 1-liner!
analysis = da.run_data_assimilation(
    model=model,
    scheduler=scheduler,
    observations=observations,
    obs_mask=obs_mask,
    obs_noise_std=0.3,
    gamma=1e-3,
    num_inference_steps=50,
    n_samples=1,
    seed=42,
)

print(f"Analysis shape: {analysis.shape}")

In [ ]:
# Denormalize for plotting
truth_np = truth[0, 0].cpu().numpy() * (train_ds.max - train_ds.min) + train_ds.min
analysis_np = analysis[0, 0].cpu().numpy() * (train_ds.max - train_ds.min) + train_ds.min

# Masked observations for plotting
obs_plot = truth_np.copy()
mask_2d = obs_mask[0, 0].numpy()
obs_plot[~mask_2d] = np.nan

# Compute metrics
diff = analysis_np - truth_np
rmse = np.sqrt(np.mean(diff**2))
corr = np.corrcoef(truth_np.ravel(), analysis_np.ravel())[0, 1]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
vmin, vmax = truth_np.min(), truth_np.max()

im0 = axes[0].imshow(truth_np, cmap='viridis', vmin=vmin, vmax=vmax)
axes[0].set_title('Ground Truth')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(obs_plot, cmap='viridis', vmin=vmin, vmax=vmax)
oy, ox = np.where(mask_2d)
axes[1].scatter(ox, oy, c='red', s=5, alpha=0.7)
axes[1].set_title(f'Observations (10%)')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(analysis_np, cmap='viridis', vmin=vmin, vmax=vmax)
axes[2].set_title(f'DA Analysis\nRMSE={rmse:.3f}, r={corr:.3f}')
axes[2].axis('off')
plt.colorbar(im2, ax=axes[2], fraction=0.046)

im3 = axes[3].imshow(np.abs(diff), cmap='magma')
axes[3].set_title('|Analysis - Truth|')
axes[3].axis('off')
plt.colorbar(im3, ax=axes[3], fraction=0.046)

plt.suptitle('Step 3: Score-Based Data Assimilation', fontsize=14)
plt.tight_layout()
plt.show()

### DA with Different Observation Densities

Compare reconstruction quality with 5%, 10%, 30%, and 50% observations.

In [ ]:
obs_fractions = [0.05, 0.10, 0.30, 0.50]
fig, axes = plt.subplots(2, len(obs_fractions) + 1, figsize=(20, 8))
vmin, vmax = truth_np.min(), truth_np.max()

# Ground truth column
for row in range(2):
    axes[row, 0].imshow(truth_np, cmap='viridis', vmin=vmin, vmax=vmax)
    axes[row, 0].axis('off')
axes[0, 0].set_title('Ground Truth')
axes[1, 0].set_title('Ground Truth')

rmses, corrs = [], []

for j, frac in enumerate(obs_fractions):
    mask = da.make_random_mask((1, 1, H, W), obs_fraction=frac, seed=42)
    obs = truth.clone()
    obs[~mask.to(device)] = float('nan')

    result = da.run_data_assimilation(
        model=model, scheduler=scheduler,
        observations=obs, obs_mask=mask,
        obs_noise_std=0.3, gamma=1e-3,
        num_inference_steps=50, seed=42,
    )

    res_np = result[0, 0].cpu().numpy() * (train_ds.max - train_ds.min) + train_ds.min
    diff = res_np - truth_np
    r = np.sqrt(np.mean(diff**2))
    c = np.corrcoef(truth_np.ravel(), res_np.ravel())[0, 1]
    rmses.append(r)
    corrs.append(c)

    # Observations row
    obs_vis = truth_np.copy()
    m2d = mask[0, 0].numpy()
    obs_vis[~m2d] = np.nan
    axes[0, j+1].imshow(obs_vis, cmap='viridis', vmin=vmin, vmax=vmax)
    oy, ox = np.where(m2d)
    axes[0, j+1].scatter(ox, oy, c='red', s=3, alpha=0.5)
    axes[0, j+1].set_title(f'Obs ({frac:.0%})')
    axes[0, j+1].axis('off')

    # Analysis row
    axes[1, j+1].imshow(res_np, cmap='viridis', vmin=vmin, vmax=vmax)
    axes[1, j+1].set_title(f'Analysis\nRMSE={r:.3f}, r={c:.3f}')
    axes[1, j+1].axis('off')

plt.suptitle('DA Quality vs Observation Density', fontsize=14)
plt.tight_layout()
plt.show()

# Summary plot
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(obs_fractions, rmses, 'bo-', label='RMSE')
ax1.set_xlabel('Observation Fraction')
ax1.set_ylabel('RMSE', color='b')
ax2 = ax1.twinx()
ax2.plot(obs_fractions, corrs, 'rs-', label='Correlation')
ax2.set_ylabel('Correlation', color='r')
ax1.set_title('Reconstruction Quality vs Observation Density')
ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.show()

## Next Steps

- **Real weather data**: See `01_training_diffusion_priors_weather.ipynb` for NCEP reanalysis
- **Longer NAS**: Use `time_budget_seconds=300` and the default search space (`search_space=None`) for thorough architecture search
- **More epochs**: Train for 100-1000 epochs for better results on real data
- **Tune DA params**: Experiment with `obs_noise_std` and `gamma` for your dataset
- **Multiple schedulers**: Try `heun_edm`, `ddim`, `euler`, `dpm_solver` via `da.build_scheduler()`